# 02 – Hypothesis Testing & A/B Analysis

In this notebook we statistically test the following null hypotheses:

1. There are no risk differences across provinces.
2. There are no risk differences between zip codes.
3. There is no significant margin (profit) difference between zip codes.
4. There is no significant risk difference between women and men.

Risk is measured by:
- Claim frequency (`HasClaim`)
- Claim severity (`TotalClaims` where `HasClaim == 1`)
- Margin (`TotalPremium - TotalClaims`)


In [ ]:
import os

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

from src.data.load_data import load_processed_data
from src.features.build_features import add_core_features

pd.set_option("display.max_columns", 100)

## 1. Load the processed dataset

In [ ]:
df = load_processed_data()
df = add_core_features(df)
df.head()

## 2. Helper: print test result

In [ ]:
def print_test_result(name: str, statistic: float, p_value: float, alpha: float = 0.05):
    decision = "REJECT H0" if p_value < alpha else "Fail to reject H0"
    print(f"\n=== {name} ===")
    print(f"Test statistic: {statistic:.4f}")
    print(f"p-value:        {p_value:.4g}")
    print(f"Decision (@ alpha={alpha}): {decision}")

## 3. Risk differences across provinces

In [ ]:
if {"Province", "HasClaim"}.issubset(df.columns):
    contingency = pd.crosstab(df["Province"], df["HasClaim"])
    chi2, p, dof, expected = stats.chi2_contingency(contingency)
    print_test_result("Chi-square test – Claim frequency by Province", chi2, p)

    # Severity ANOVA: only policies with claims
    df_claims = df[df["HasClaim"] == 1].copy()
    if {"Province", "TotalClaims"}.issubset(df_claims.columns):
        model = smf.ols("TotalClaims ~ C(Province)", data=df_claims).fit()
        anova_table = sm.stats.anova_lm(model, typ=2)
        display(anova_table)
else:
    print("Required columns not found for province tests.")

## 4. Risk & margin differences between zip codes

In [ ]:
if {"PostalCode", "HasClaim"}.issubset(df.columns):
    top_zips = df["PostalCode"].value_counts().head(20).index
    df_zip = df[df["PostalCode"].isin(top_zips)].copy()

    contingency_zip = pd.crosstab(df_zip["PostalCode"], df_zip["HasClaim"])
    chi2_zip, p_zip, *_ = stats.chi2_contingency(contingency_zip)
    print_test_result("Chi-square test – Claim frequency by PostalCode (top 20)", chi2_zip, p_zip)

    # Severity ANOVA
    df_zip_claims = df_zip[df_zip["HasClaim"] == 1]
    if "TotalClaims" in df_zip_claims.columns:
        model_zip = smf.ols("TotalClaims ~ C(PostalCode)", data=df_zip_claims).fit()
        anova_zip = sm.stats.anova_lm(model_zip, typ=2)
        display(anova_zip)

    # Margin ANOVA
    if "Margin" in df_zip.columns:
        model_margin = smf.ols("Margin ~ C(PostalCode)", data=df_zip).fit()
        anova_margin = sm.stats.anova_lm(model_margin, typ=2)
        display(anova_margin)
else:
    print("Required columns not found for zip/postal code tests.")

## 5. Risk differences between women and men

In [ ]:
if {"Gender", "HasClaim"}.issubset(df.columns):
    # Claim frequency – chi-square test
    gender_tab = pd.crosstab(df["Gender"], df["HasClaim"])
    chi2_gender, p_gender, *_ = stats.chi2_contingency(gender_tab)
    print_test_result("Chi-square test – Claim frequency by Gender", chi2_gender, p_gender)

    # Severity – t-test on policies with claims
    df_gender_claims = df[df["HasClaim"] == 1].copy()
    if {"Gender", "TotalClaims"}.issubset(df_gender_claims.columns):
        female = df_gender_claims[df_gender_claims["Gender"] == "Female"]["TotalClaims"]
        male = df_gender_claims[df_gender_claims["Gender"] == "Male"]["TotalClaims"]

        t_stat, p_ttest = stats.ttest_ind(female, male, equal_var=False, nan_policy="omit")
        print_test_result("t-test – Claim severity Female vs Male", t_stat, p_ttest)
else:
    print("Required columns not found for gender tests.")

## 6. Interpreting the results

For each hypothesis, look at:

- The p-value (is it below 0.05?)
- The direction and size of the differences (e.g. which province has higher loss ratio?)
- What this means for pricing and marketing.

You can summarise these interpretations in your final report under
the "Hypothesis Testing" section.
